In [2]:
import pandas as pd
import glob
import time
import duckdb
import sqlite3
import matplotlib

In [3]:
conn =  duckdb.connect('taxi.db')

Execute this line below if there is no taxi.db

In [6]:
df = conn.execute ("""
    CREATE TABLE IF NOT EXISTS trips AS
    SELECT * 
    FROM "../data/raw/yellow/*.parquet"
""").df()

In [9]:
conn.execute("SELECT * FROM trips LIMIT 5").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [7]:

conn.execute("SELECT COUNT(*) FROM trips").df()

,count_star()
0,48722602


In [17]:
conn.execute("DESCRIBE trips").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [4]:
conn.execute("""
    SELECT 
        MAX(tpep_dropoff_datetime) AS max_dropoff,
        MIN(tpep_pickup_datetime) AS min_pickup,
        max(tpep_dropoff_datetime) - min(tpep_pickup_datetime) AS date_range,
        MAX(tpep_dropoff_datetime - tpep_pickup_datetime) AS max_trip_duration,
        MIN(tpep_dropoff_datetime - tpep_pickup_datetime) AS min_trip_duration,
        SUM (CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS missing_pickup_datetime,
        SUM (CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS missing_dropoff_datetime,
        SUM (CASE WHEN (tpep_dropoff_datetime < tpep_pickup_datetime) THEN 1 ELSE 0 END) AS impossible_duration,
        SUM (CASE WHEN (tpep_dropoff_datetime = tpep_pickup_datetime) THEN 1 ELSE 0 END) AS flash_duration
    FROM trips
""").df()

,max_dropoff,min_pickup,date_range,max_trip_duration,min_trip_duration,missing_pickup_datetime,missing_dropoff_datetime,impossible_duration,flash_duration
0,2026-01-05 13:00:34,2007-12-05 18:45:00,6605 days 18:15:34,10 days 08:00:46,-36 days +06:07:41,0.0,0.0,2235.0,544069.0


In [ ]:
conn.execute("""
    SELECT
        SUM (CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS missing_trip_distance,
        SUM (CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS missing_pickup_location,
        SUM (CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS missing_dropoff_location,
        MAX (trip_distance) AS max_trip_distance,
        MIN (trip_distance) AS min_trip_distance,
        SUM (CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS missing_passenger_count,
        MAX (passenger_count) AS max_passenger_count,
        MIN (passenger_count) AS min_passenger_count,
        AVG (passenger_count) AS avg_passenger_count,
        SUM (CASE WHEN passenger_count == 0 THEN 1 ELSE 0 END) AS zero_passenger_count,
        SUM (CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fares
    FROM trips;
""").df()

,missing_trip_distance,missing_pickup_location,missing_dropoff_location,max_trip_distance,min_trip_distance,missing_passenger_count,max_passenger_count,min_passenger_count,avg_passenger_count,zero_passenger_count,negative_fares,unique_vendors
0,0.0,0.0,0.0,397994.37,0.0,11611894.0,9,0,1.295491,260062.0,2848620.0,4


In [10]:
conn.execute("""
    WITH row_counts AS (
    SELECT
        *,
        COUNT(*) AS row_count
    FROM trips
    GROUP BY ALL
)

SELECT
    SUM(row_count - 1) AS duplicate_row_count
FROM row_counts
WHERE row_count > 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_row_count
0,1.0


In [11]:
conn.close()